# bfs-gpu on Google Colab (Group 17)

Builds and checks the project on a Colab GPU. **Before running:** *Runtime -> Change runtime type -> T4 GPU*,
then *Runtime -> Run all*. Each step says what output to expect.

Step 2 clones the team repository https://github.com/yash-602/bfs-gpu (public). Without internet access to
GitHub, use the zip-upload cell instead (`python scripts/make_zip.py` in the repo makes `bfs-gpu.zip`).

**Warning (playbook 4.3):** Colab GPUs differ between sessions and the disk is wiped when the session ends.
Use any Colab session for development. If Colab is the reference machine, run the *final* experiments in
**one single session**, and save `results/` (including `env.txt` and `gpu_info.txt`) to Google Drive or
download it before closing.

## 1. GPU and toolchain

In [ ]:
!nvidia-smi
!nvcc --version | tail -2
!g++ --version | head -1

## 2. Get the code
Clones the latest `main` of the team repository. (Re-running this cell gets a fresh copy.)

In [ ]:
!rm -rf /content/bfs-gpu && git clone -q https://github.com/yash-602/bfs-gpu.git /content/bfs-gpu
!git -C /content/bfs-gpu log --oneline -3

In [ ]:
# ALTERNATIVE to the cell above (only if cloning is not possible): upload bfs-gpu.zip instead
# import os, shutil, zipfile
# from google.colab import files
# os.chdir('/content')
# up = files.upload()                      # choose bfs-gpu.zip
# shutil.rmtree('/content/bfs-gpu', ignore_errors=True)
# zipfile.ZipFile(next(iter(up))).extractall('/content/bfs-gpu')

## 3. Pick the CUDA architecture of this GPU
A T4 is compute capability 7.5, so `ARCH=sm_75`. `make` reads `ARCH` from the environment, so every later
step (including `scripts/check_all.sh`) builds for this GPU.

In [ ]:
import os, subprocess
cap = subprocess.check_output(['nvidia-smi', '--query-gpu=compute_cap', '--format=csv,noheader']).decode().split()[0]
os.environ['ARCH'] = 'sm_' + cap.replace('.', '')
os.chdir('/content/bfs-gpu')
print('GPU compute capability', cap, '-> ARCH =', os.environ['ARCH'])
# While Member 3's adaptive program is still a placeholder, benchmark only serial + naive.
adaptive_ready = 'PLACEHOLDER-NOT-YET-DELIVERED' not in open('src/bfs_adaptive.cu').read()
os.environ['IMPL'] = 'both' if adaptive_ready else 'naive'
print('adaptive program delivered:', adaptive_ready, '-> IMPL =', os.environ['IMPL'])

## 4. Host build and unit tests (Member 1)
Expected last line: `39 checks passed, 0 failed`.

In [ ]:
%%bash
cd /content/bfs-gpu
make clean > /dev/null
make host 2>&1 | grep -iE "warning|error" || echo "make host: no warnings"
./bin/test_host

## 5. Toolchain check: `hello_cuda`
Expected: a `GPU: ...` line, then `hello_cuda: PASS`.

In [ ]:
%%bash
cd /content/bfs-gpu
nvcc -O2 -std=c++17 -arch=$ARCH -Iinclude tests/hello_cuda.cu -o /tmp/hello && /tmp/hello

## 6. Golden values (playbook 6.7)
Expected: `n=262144 m=7612566`; stats show 88105 isolated vertices, max degree 25096, CV 6.473,
est. diameter 7; serial first line `src=149623 ... levels=6 edges=7612472 ... structural=OK`.

In [ ]:
%%bash
cd /content/bfs-gpu
./bin/graph_tool rmat 18 16 1 /tmp/r18.csr
./bin/graph_tool stats /tmp/r18.csr | sed -n '1,11p'
./bin/graph_tool serial /tmp/r18.csr 2

## 7. Full check: `scripts/check_all.sh`
Step 4 runs the naive GPU tests (12 configurations): every line must say PASS. While Member 3's file is still
a placeholder, step 5 prints `TEST_ADAPTIVE: FAIL (placeholder)` and the script stops there; that is expected.
Once Member 3 has delivered, the expected last line is `ALL CHECKS PASSED`.

In [ ]:
%%bash
cd /content/bfs-gpu
bash scripts/check_all.sh 2>&1 | grep -vE "^(g\+\+|nvcc) "

## 8. Smoke test of the benchmark
Expected last line: `all runs verified OK -> /tmp/smoke.csv` (written to /tmp so it never mixes with real results).

In [ ]:
%%bash
cd /content/bfs-gpu
mkdir -p data/csr
[ -f data/csr/rmat16.csr ] || ./bin/graph_tool rmat 16 16 1 data/csr/rmat16.csr
rm -f /tmp/smoke.csv
./bin/bench --graph data/csr/rmat16.csr --category rmat --impl $IMPL --sources 2 --trials 2 \
            --out /tmp/smoke.csv --structural
head -3 /tmp/smoke.csv

## 9. Datasets (needed for step 11)
Downloads about 2 GB from SNAP and converts everything (roughly 10-15 minutes). Colab's disk is wiped at
the end of the session; to keep the `.csr` files, mount Drive in the next cell first. Converting com-orkut
needs about 4.5 GB of RAM.

In [ ]:
# Optional: keep data/ on Google Drive across sessions (asks for permission to access your Drive)
# from google.colab import drive
# drive.mount('/content/drive')
# !mkdir -p /content/drive/MyDrive/bfs-gpu-data && rm -rf /content/bfs-gpu/data \
#   && ln -s /content/drive/MyDrive/bfs-gpu-data /content/bfs-gpu/data

In [ ]:
%%bash
cd /content/bfs-gpu
bash scripts/download_datasets.sh 2>&1 | grep -E "^\[|done"
bash scripts/convert_all.sh 2>&1 | grep -E "^\[|parsed|generated|missing"

## 10. Record the machine (do this in the session that produces final numbers)
Writes `results/env.txt` and `results/gpu_info.txt`, which must sit next to the final CSVs.

In [ ]:
%%bash
cd /content/bfs-gpu
mkdir -p results
bash scripts/env_report.sh > results/env.txt
nvidia-smi > results/gpu_info.txt
[ -x bin/bench ] && ./bin/bench --gpu-info >> results/gpu_info.txt || true
cat results/env.txt

## 11. Experiments (Member 2's `run_all.sh`) and analysis
Needs the datasets from step 9. **Quick run** (default below): 2 sources x 2 trials, E2 + E5 + E6, to check the
whole pipeline; roughly tens of minutes on a T4. **Final run**: set `SRC=5 TRIALS=5` and add `E4` (and `E3`
once the adaptive program exists), all in one session. Progress lines start with `>>>`; any
`VERIFICATION FAILED` line is a bug to report to the owner of that implementation.

In [ ]:
%%bash
cd /content/bfs-gpu
rm -f results/E*.csv            # start clean so old rows never mix with new ones
SRC=2 TRIALS=2 bash scripts/run_all.sh E2 E5 E6 2>&1 | grep -E "^>>>|verified|FAILED|error|finished"

In [ ]:
%%bash
cd /content/bfs-gpu
pip -q install pandas matplotlib numpy > /dev/null
python3 scripts/analyze.py 2>&1 | grep -v FutureWarning
ls results/figs

In [ ]:
# show the main table and figures inline
from IPython.display import Markdown, Image, display
display(Markdown(open('/content/bfs-gpu/results/tables.md').read()))
import glob
for p in sorted(glob.glob('/content/bfs-gpu/results/figs/*.png')):
    print(p); display(Image(p))

## 12. Save results
Downloads `results/` as a zip (CSV files, figures, env.txt, gpu_info.txt).

In [ ]:
import shutil
from google.colab import files
shutil.make_archive('/content/results', 'zip', '/content/bfs-gpu/results')
files.download('/content/results.zip')